In [38]:
# imports

# show rich formats 
from IPython.display import display, Markdown

# get python to interact with openai
from openai import OpenAI

# use personal openai key
import os
from dotenv import load_dotenv

# check load_dotenv works - should come back 'True'
# load_dotenv()

# make a destination 'resumes' directory for the work

os.makedirs("resumes", exist_ok=True)

# use python to format markdown to html
from markdown import markdown

#### because we're sending this out as a resume, we want it to be in a .pdf format. 
#### that means we'll have to use a library called weasy... 
##### <i><b>< record scratch ></i></b>
#### nope. sorry. 
#### not using weasyprint. lining weasyprint libraries up with each particular python environmet and setting / dedicating kernals still causes nightmares of 'public-speaking-in-underpants' proportions. surely a powerful tool, but it's got the playfulness and ease of use of a rabid porcupine. <i>no grazie</n>.
#### instead, going with pdfkit. working in html, so it'll do the job.
##### * note: pdfkit works using wkhtmltopdf, which <i>in very simple terms</i> converts a webpage to a pdf file. please see [reference.txt](https://github.com/npj210mlk/jobapp_prompter/blob/main/requirements.txt) in the repo for installation instructions.


In [39]:
# import pdfkit

# # test
# pdfkit.from_string("<h1>It should be called 'QueezyPrint,' amirite?</h1>", "output.pdf")

#### ha! apparently, jupyter agrees - came back 'True'

***
#### with the libraries imported we can now break the project down into four (4) steps:
##### 1.) open and read the markdown resume file
#####     * see requirements notes
##### 2.) input the desired job description
##### 3.) template some prompt engineering
##### 4.) convert markdown to html
##### 5.) convert html to pdf
***
*** 
#### Step 1: Open and Read the Markdown Resume
****

In [42]:
# open resume and read it
with open("/Users/nicholasjoseph/Desktop/nj_pm_res.md", "r", encoding = "utf-8") as resume_file:
    resume_string = resume_file.read()

# check to see if it worked:
# display(Markdown(resume_string))

# markdown resume displays as expected.

***
#### Step 2: Input the Desired Job Description
***

In [43]:
# job description will be from anywhere, so input is used to pause the program until we find one to copy/paste into this variable 
job_desc_string = input()

 About the job ClassDojo's goal is to give every child on Earth an education they love.   We started by building a powerful network for communication. ClassDojo’s flagship app is the #1 communication app connecting K-12 teachers, children, and families globally. Teachers use it to share what’s happening throughout the day through photos, videos, and messages that make parents feel like they’re there. It’s actively used in over 95% of US schools, reaching over 45 million children in 180 countries, with a team of just around 200 people [1]. We are now beginning to use this network to give kids the best learning experiences in the world, far beyond those a standard school can provide.   We hire for talent density. Our team comprises the most talented, entrepreneurial, and innovative teammates from around the world, with experience in education and large scale consumer internet companies, including Instagram, Netflix, Dropbox, Stripe, Uber, Y Combinator, and more. We’re building a company 

***
#### Step 3: Template Some Prompt Engineering
##### - this involves dealing with ChatGPT, particularly the ChatGPT 4o-mini pre-trained transformer.
##### <u>a couple of things about the ChatGPT 4o-mini engine (model)</u>:
#####     a.) it is the smaller, more affordable version of the massive GPT-4o engine available to developers; and
#####     b.) because its focus is on text classificaton / prediction, it is purely a "decoder-only" type transformer
#### The idea is to have ChatGPT create the prompt to respond to the likely Applicant Tracking System being used by the job poster.
##### Reason being, machines talk to machines better than we can. 
***

In [44]:
# making up a random "lambda" function to create the variable "prompt_template"

# the text is what I would be putting into ChatGPT were I to do this one job at a time - prompt engineering

prompt_template = lambda resume_string, job_desc_string : f"""
### Role: 
You are a professional resume optimization expert, tailoring my resume to fit specific job descriptions. 
You know my job preferences include collaborating with people, and helping businesses get the most out of their data.
Your goal is to optimize my resume and provide actionable suggestions for improvement to align with the target role.

### Guidelines:
1. **Relevance**:
    - Prioritize the particular skills and experiences I have with what is **most relevant to the job position**.
    - De-emphasize or even completely remove irrelevant details to ensure a **concise** and **targeted** resume.
    - Limit work experience section to 2-3 most relevant roles
    - Limit bullet points under each role to 2-3 most relevant impacts

2. **Action-Driven Results**:
    - Choose **strong action verbs** and **quantifiable results** (eg: percentages, revenues, efficiency improvement, etc.)

3. **Keyword Optimization**:
    - Integrate **keywords** and phrases from the job description naturally to optimize for Applicant Tracking Systems (ATS)

4. **Additional Suggestions*** *(if gaps exist)*:
    - If the resume does not fully align with the job description, suggest:
        a.) **Additional technical or soft skills** that I could add to make my profile stronger.
        b.) **Certifications or courses** I have (or could pursue) that would bridge the gap(s).
        c.) **Project ideas or experiences** that would better align with the role.

5.) **Formatting**:
    - Ouptut the tailored resume in **clean Markdown format**.
    - Include an **"Additional Suggestions"** section at the end with actionable improvement recommendations.

---

## Input:
- **My resume**:
{resume_string}

- **The Job Description**:
{job_desc_string}

---

### Output:
1. **Tailored Resume**:
    - A resume in **Markdown format** that emphasizes relevant experience, skills, and achievements.
    - Incorporates job description **keywords** to optimize for ATS.
    - Uses confident language and is no longer than **one page**.

2. **Additional Suggestions** *(if applicable)*:
    - List **skills** that could strengthen alignment with the role.
    - Recommend **certifications or courses** to pursue.
    - Suggest **specific projects or experiences** to develop.
"""

In [45]:
# set the prompt variable for our ChatGPT message roles
prompt = prompt_template(resume_string, job_desc_string)

***
#### Step 4: Generate the Resume with GPT-4o-mini
##### - Make the api call and tell GPT to write the resume using the prompt from above
***

In [46]:
# set up api client
open_apikey = os.environ.get("openapi_apikey")
    
client = OpenAI(api_key = open_apikey)

# make the call

# set response variable to hold the all info we get back
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    
    # set our roles up - think of casting a play: "Today, the role of the Expert Resume Writer will be played by the system."
    messages = [
        {"role" : "system", "content" : "Expert Resume Writer"},
        {"role" : "user", "content" : prompt}
    ],
    
    # give it some creative license 
    temperature = 0.7
)

# get our response
response_string = response.choices[0].message.content

***
#### Step 5: Show Us the Results
***

In [47]:
# separate new resume from improvements AI suggests at 'Additional Suggestions'
response_list = response_string.split("## Additional Suggestions")

In [52]:
display(Markdown(response_list[0]))

# Nicholas Joseph

**Product Manager**

San Antonio, TX | Remote | Hybrid | Local to San Antonio, TX  
nickpjoseph210@gmail.com | (210) 771-9853 | [LinkedIn](https://www.linkedin.com/in/nickjosephsanantonio/) | [GitHub](https://github.com/npj210mlk)

## Career Summary

Dynamic Product Manager with 7+ years of experience creating and scaling consumer products in Agile environments. Proven ability to translate complex data into user-friendly experiences, with a strong focus on enhancing product strategies through user insights. Background in data engineering and analytics, enabling data-driven decisions that improve user engagement and drive business growth.

## Impact Summary

* **NobleHands:** Founded and managed a residential construction business, achieving 200% year-over-year growth by identifying and addressing customer needs.
* **Spaulding Ridge:** Delivered data pipelines for the NFL's Fan 360 Experience, enhancing data efficiency by 98.9% and enabling better product insights and user experience.
* **Apexon:** Established a mentorship program, reducing new employee turnover by over 30%, fostering collaboration and improving team dynamics.

## Professional Experience

**Cloud Data Engineer** | Spaulding Ridge | Remote | 07/2022-06/2023

* Led migration of Seattle Mariners' data systems to GCP, enhancing analytics for product decision-making and user engagement.
* Collaborated with cross-functional teams, providing insights that influenced strategic initiatives and product development.
* Developed training materials to facilitate data democratization, empowering users to leverage product capabilities effectively.

**Project Manager/Owner** | NobleHands H & C | San Antonio, TX | 10/2017-01/2020, 07/2023-Present

* Founded and managed a residential construction business, focusing on customer satisfaction and service delivery, leading to 200% growth.
* Resolved client conflicts through effective communication and active listening, ensuring mutually beneficial outcomes and enhancing customer loyalty.

**Junior Data Engineer** | Apexon | Remote | 12/2020-03/2022

* Contributed to the migration of a federal bank's data systems, improving operational efficiency and facilitating user access to vital information.
* Co-created the Gathi Garden Program, mentoring new hires and improving onboarding processes, which enhanced team collaboration and productivity.

## Certifications & Education

* Professional Scrum Master I | scrum.org, 12/2024
* Project Management Professional (in progress) | Technical Institute of America, 2024
* Data Science Certificate | Codeup, San Antonio TX | 2020
* B.S. in Biology | Concordia University, Austin TX

## Technical Skills

**Programming Languages:** Python3, SQL, HTML, Spark  
**Cloud Platforms:** GCP, Snowflake, Azure, AWS  
**Data Visualization:** Tableau, Looker  
**Methodologies:** Agile (Scrum/Kanban), Scaled Agile (SAFe)  
**Other:** Data Modeling, User Research & Usability Testing

---



***
#### Step 6: Save the New Resume
##### Great. Hit several snags. You either need WeasyPrint installed in some capacity, or other tools I found (like 'Grip') are difficult to automate and test on MacOS. 
##### Looks like the programming gods have spoken: we're going with WeasyPrint.
##### <center><span style ="color:red"><b><u>To Do This:</u></b></span><center>
##### 1.) completely uninstall existing WeasyPrint's existence from your machine with pip and brew uninstalls;
##### 2.) run a 'brew cleanup' just to wipe out any remnants;
##### 3.) form home directory in Terminal (I'm using Mac), type 'brew install cairo pango gdk-pixbuf libffi' - these are the native Weasyprint dependencies;
##### 4.) set your environment variables in your profile (for me, my ~/.zshrc file) to the following:
###### export DYLD_LIBRARY_PATH=/opt/homebrew/lib:$DYLD_LIBRARY_PATH
###### export LIBRARY_PATH="/opt/homebrew/lib:$LIBRARY_PATH"
###### export PKG_CONFIG_PATH="/opt/homebrew/lib/pkgconfig:/opt/homebrew/share/pkgconfig"
###### export PATH="/opt/homebrew/bin:$PATH"
##### 5.) restart the terminal;
##### 6.) navigate to your project folder;
##### 7.) type 'pip install weasyprint markdown'; and (finally)
##### 8.) open your notebook from your project file with 'jupyter notebook'
***

In [53]:
# Markdown was already imported earlier
# import WeasyPrint and its HTML abilities
from weasyprint import HTML

# save it as PDF
output_pdf_file = "resumes/nick_joseph_tam_resume.pdf"

# convert the Markdown file to HTML
html_content = markdown(response_list[0])

# now take that HTML and convert it into a PDF and save
HTML(string=html_content).write_pdf(output_pdf_file)

In [54]:
# let's save the new Markdown file, too
markdown_output = "resumes/resume_new_pm_resume_markdown.md"

with open(markdown_output, "w", encoding = "utf-8") as file:
    file.write(response_list[0])

***
#### Step 7: Display Suggestions for Improvement
##### Because we split "response_string" earlier, it was split at "Additional Suggestions," giving two items "response_list."
##### The first item ([0]) is the resume ChatGPT wrote with our prompt.
##### The second item ([1]) are the suggestions ChatGPT offers to make our resume stronger
***

In [55]:
# show us how to make the resume stronger
display(Markdown(response_list[1]))



1. **Skills to Add**:
   - User experience design, qualitative research methodologies, product monetization strategies.
   - Familiarity with subscription-based models and marketplace dynamics.

2. **Certifications or Courses**:
   - Consider pursuing a course in Product Management from a recognized platform (e.g., Coursera, Udacity).
   - Explore certifications in User Experience (UX) or Design Thinking to enhance user-centric product development skills.

3. **Project Ideas**:
   - Develop a personal project focusing on creating a prototype for an educational tool aimed at improving student engagement.
   - Volunteer to mentor or work with educational organizations to gain insights into the specific needs of K-12 environments and the challenges they face.

In [13]:
# # markdown is already imported
# from weasyprint import HTML

In [14]:
# from markdown2pdf import convert

# markdown_resume = display(Markdown(response_list[0]))

# convert(markdown_resume, "nj_resume_in_pdf.pdf")